In [2]:
import pandas as pd
from collections import defaultdict

d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')

df = d[['Smell_Word', 'year']].copy()
df['year'] = pd.to_numeric(df['year'], errors='coerce')

df = df.dropna(subset=['Smell_Word', 'year'])

word_freq_per_year = defaultdict(lambda: defaultdict(int))

for _, row in df.iterrows():
    year = int(row['year'])
    words = str(row['Smell_Word']).lower().split('|')  # Split sulle pipe
    for word in words:
        word = word.strip()
        if year and word:
            word_freq_per_year[year][word] += 1

records = []
for year, freqs in word_freq_per_year.items():
    for word, freq in freqs.items():
        records.append({'word': word, 'frequency': freq, 'year': year})

df_word_freq_year = pd.DataFrame(records)

df_word_freq_year = df_word_freq_year.sort_values(['year', 'frequency'], ascending=[True, False])

# df_word_freq_year


/var/folders/j8/2fw1pn3n4zb1y5l8gt8s56dc0000gn/T/ipykernel_64146/2361456401.py:4: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_table('./df_new_all_lemma.tsv', sep='\t', on_bad_lines='skip')


In [3]:
word_freq_per_year = defaultdict(lambda: defaultdict(int))
for _, row in df.iterrows():
    year = int(row['year'])
    words = str(row['Smell_Word']).lower().split('|')
    for word in words:
        word = word.strip()
        if year and word:
            word_freq_per_year[year][word] += 1

records = []
for year, freqs in word_freq_per_year.items():
    for word, freq in freqs.items():
        records.append({'word': word, 'frequency': freq, 'year': year})

df_word_freq_year = pd.DataFrame(records).sort_values(['year', 'frequency'], ascending=[True, False])

In [4]:
df_word_freq_year = df_word_freq_year[df_word_freq_year['year'] < 2000]
df_word_freq_year = df_word_freq_year[df_word_freq_year['year'] > 1600]
df_word_freq_year

,word,frequency,year
66264,smell,75,1601
66270,odoriferous,29,1601
66267,stink,27,1601
66268,perfume,17,1601
66274,odour,15,1601
...,...,...,...
72290,toothpaste,1,1999
72295,perfumes,1,1999
72296,essential,1,1999
72297,aromas,1,1999


In [5]:
from collections import defaultdict

def calculate_relative_frequency_per_word_df(df, categories):
    word_frequency = defaultdict(lambda: defaultdict(float))
    total_per_year = df.groupby('year')['frequency'].sum().to_dict()
    
    for _, row in df.iterrows():
        year = int(row['year'])
        word = row['word'].lower()
        freq = row['frequency']
        total = total_per_year[year]
        
        for category, category_words in categories.items():
            if word in category_words:
                word_frequency[word][year] += freq / total
    
    return word_frequency


def relative_frequency_per_word_df(word_frequency):
    for category, words in categories.items():
        print(f"--{category}--")
        for word in words:
            if word in word_frequency:
                print(f"Relative frequency for the word '{word}':")
                for year, frequency in sorted(word_frequency[word].items()):
                    print(f"    Year {year}: {frequency:.2%}")
                print()


categories = {  
    'stench/stinking': ['stink', 'stinch', 'stench', 'reek', 'whiff', 'fetor', 'foetor', 'redolence', 'pong', 'niff', 'pungency', 'stinking', 'malodorous', 'fetid', 'foetid', 'niffy', 'smelly', 'reeking', 'whiffy', 'pungent', 'noisome', 'funky', 'musty', 'frowzy'],
    'fragrance/fragrant': ['redolence', 'perfume', 'scent', 'aroma', 'fragrance', 'musk', 'scented', 'aromatic', 'fragrant', 'redolent', 'sweet', 'fragrancy', 'odoriferousness'],
    'lacking_odour': ['odourless', 'odorless', 'scentless', 'unscented', 'deodorized', 'deodorization', 'deodorizer', 'deodorant', 'unsmelling', 'savourless', 'inodorate']
}


relative_word_frequency_per_year = calculate_relative_frequency_per_word_df(
    df_word_freq_year,
    categories
)

#relative_frequency_per_word_df(relative_word_frequency_per_year)

In [6]:
def print_top_words_per_category_per_year_raw(data, categories, top_n=10):
    """
    data: dict -> word -> year -> absolute frequency (raw count)
    categories: dict -> category -> list of words
    """
    for category, category_words in categories.items():
        years = set()
        for word in category_words:
            if word in data:
                years.update(data[word].keys())
        years = sorted(years)

        for year in years:
            category_words_year = {
                word: data[word][year]
                for word in category_words
                if word in data and year in data[word]
            }
            
            if not category_words_year:
                continue
            
            sorted_words = sorted(
                category_words_year.items(),
                key=lambda x: x[1],
                reverse=True
            )[:top_n]
            
            print(
                f"\nTop {top_n} words in the category "
                f"'{category}' for the year {year}:"
            )
            
            for word, freq in sorted_words:
                print(f"    {word}: {freq:.2%}")


# print_top_words_per_category_per_year_raw(
#     relative_word_frequency_per_year,
#     categories,
#     top_n=10
# )

In [7]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'fragrance/fragrant'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())

rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")


Spearman rho for entropy over time: 0.344, p = 0.000
Spearman rho for dominance over time: -0.127, p = 0.011


In [8]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'stench/stinking'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())


rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")


Spearman rho for entropy over time: 0.665, p = 0.000
Spearman rho for dominance over time: -0.702, p = 0.000


In [9]:
import numpy as np
from scipy.stats import spearmanr

# Select the category to analyze
category = 'lacking_odour'
category_words = categories[category]


years = sorted({
    year
    for word in category_words
    if word in relative_word_frequency_per_year
    for year in relative_word_frequency_per_year[word]
})

entropy_per_year = []
dominance_per_year = []

for year in years:

    freqs = [
        relative_word_frequency_per_year[word][year]
        for word in category_words
        if word in relative_word_frequency_per_year
        and year in relative_word_frequency_per_year[word]
    ]
    
    freqs = np.array(freqs)
    freqs = freqs / freqs.sum()  # Normalize for safety
    

    H = -np.sum(freqs * np.log2(freqs + 1e-10))
    entropy_per_year.append(H)

    dominance_per_year.append(freqs.max())


rho, p = spearmanr(years, entropy_per_year)
print(f"\nSpearman rho for entropy over time: {rho:.3f}, p = {p:.3f}")


rho_d, p_d = spearmanr(years, dominance_per_year)
print(f"Spearman rho for dominance over time: {rho_d:.3f}, p = {p_d:.3f}")


Spearman rho for entropy over time: 0.279, p = 0.000
Spearman rho for dominance over time: -0.376, p = 0.000
